# Why did DAM describe the floor as a person? A small experiment

In stage 4, object 0 (the trampoline floor, the largest object) was described by DAM as *"A young person, dressed in a vibrant green shirt…"* (P44). Three possible causes, measured earlier:

1. **No zoom:** the floor is so big that its "focal crop" is the whole frame.
2. **Blurry mask:** DAM sees the mask as a 27×27 grid. Blocks on the boy's outline are half boy, half floor, so the boy partly counts as "inside the mask".
3. **Salience:** the floor is plain, while the boy is colourful and moving.

**Test:** describe the floor again under 4 conditions, each run 3 times (DAM samples its words):

| condition | frames | floor mask | if DAM now says "floor/trampoline"… |
|---|---|---|---|
| baseline | original | original | (should reproduce "person") |
| **(a)** | original | pulled back so no grid block touches the boy | …cause 2 (blurry mask) |
| **(b)** | boy painted out | original | …cause 3 (the boy draws attention) |
| **(c)** | boy painted out | pulled back | both together |

**Uses saved results only:** the stage-2 tracks, plus the same stage-3 cleanup and 8 frames as stage 4. SAM2 is not run. **Time:** DAM load (~1–2 min, more if the weights must download) + 12 descriptions × ~10 s.

In [ ]:
import os, subprocess, sys, json, time, re
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh >/dev/null 2>&1; python3 -c 'import dam' 2>/dev/null || python3 -m pip install -q --no-deps "
                   "'git+https://github.com/NVlabs/describe-anything.git'; python3 -m pip install -q sentencepiece",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
sys.path.insert(0, os.path.join(os.getcwd(), "third_party/SVG2/pipeline"))
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
import svg2_pipeline as P
OUT = "notes/floor_experiment"; os.makedirs(OUT, exist_ok=True)
FLOOR, BOY_PARTS = 0, [1, 21, 38]            # floor; the boy + his two shoe tracks (from our stage 2-4 runs)

## 1. The floor's 8 frames and masks, exactly as stage 4 used them

In [ ]:
cfg = P.PipelineConfig()
tracks = json.load(open("data/stage12/sav_000001_first120_tracks.json"))
traj = P.ObjectTrajectories(P.stage3_cleanup(cfg, tracks))
frames_all, fps, H, W = P.read_video_frames("third_party/SVG2/pipeline/examples/sav_000001.mp4")
FR = traj.sample_frames(FLOOR, cfg.caption.frames_per_object, cfg.caption.frame_sampling)
frames = {f: frames_all[f] for f in FR}; del frames_all
floor = {f: traj.mask(FLOOR, f) for f in FR}
boy = {}
for f in FR:
    parts = [traj.mask(i, f) for i in BOY_PARTS if traj.mask(i, f) is not None]
    boy[f] = np.any(parts, axis=0) if parts else np.zeros((H, W), bool)
    # parts of his legs are in no track (and the floor mask covers them), so use the convex hull of the parts
    hull = np.zeros((H, W), np.uint8)
    if boy[f].any():
        cv2.fillConvexPoly(hull, cv2.convexHull(cv2.findNonZero(boy[f].astype(np.uint8))), 1)
    boy[f] = hull > 0
print("floor frames:", FR)
saved = {d["object_id"]: d["description"] for d in json.load(open("data/stage4/sav_000001_first120_descriptions.json"))["objects"]}
print("Stage-4 description of object 0:", saved.get(FLOOR, "")[:200], "…")

## 2. Build the two changes

- **Pull the mask back (a):** remove every floor pixel within one grid block of the boy, i.e. 18 px sideways and 32 px up/down. A 384×384 view cut into 27×27 blocks makes each block about 18×31 pixels of the 480×848 frame. After this, no block contains both boy and floor, which we check.
- **Paint the boy out (b):** fill the boy region (slightly enlarged) with the floor's own median colour, then soften it. The floor is nearly one colour, so this looks like empty trampoline. OpenCV inpainting left a beige blob, which could itself look like a figure.

"Boy region" = the convex hull of ids 1, 21 and 38, because part of his legs is in no track, and the floor mask covers that part.

In [ ]:
CELL_W, CELL_H = W / 27, H / 27                       # one DAM grid block in original pixels (≈17.8 × 31.4)
k_pull = cv2.getStructuringElement(cv2.MORPH_RECT, (2 * int(np.ceil(CELL_W)) + 1, 2 * int(np.ceil(CELL_H)) + 1))
k_paint = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

def blocks(mask):
    m = cv2.resize(mask.astype(np.uint8), (384, 384), interpolation=cv2.INTER_NEAREST); p = 384 / 27
    return np.array([[m[int(i * p):int((i + 1) * p), int(j * p):int((j + 1) * p)].any() for j in range(27)] for i in range(27)])

pulled, painted = {}, {}
for f in FR:
    near_boy = cv2.dilate(boy[f].astype(np.uint8), k_pull) > 0
    pulled[f] = floor[f] & ~near_boy
    hole = cv2.dilate(boy[f].astype(np.uint8), k_paint) > 0
    img = frames[f].copy()
    img[hole] = np.median(frames[f][floor[f] & ~hole], axis=0).astype(np.uint8)   # empty-trampoline colour
    soft = cv2.GaussianBlur(img, (0, 0), 6)
    img[cv2.dilate(hole.astype(np.uint8), k_paint) > 0] = soft[cv2.dilate(hole.astype(np.uint8), k_paint) > 0]
    painted[f] = img
    shared_before = int((blocks(floor[f]) & blocks(boy[f])).sum()); shared_after = int((blocks(pulled[f]) & blocks(boy[f])).sum())
    print(f"frame {f:3d}: floor {floor[f].mean():.0%} of frame → pulled back {pulled[f].mean():.0%}; "
          f"grid blocks shared with the boy: {shared_before} → {shared_after}")

CONDITIONS = {
    "baseline":            ({f: frames[f] for f in FR},  floor),
    "(a) mask pulled back": ({f: frames[f] for f in FR},  pulled),
    "(b) boy painted out":  (painted,                      floor),
    "(c) both":             (painted,                      pulled),
}
f0 = FR[4]
fig, axes = plt.subplots(2, 4, figsize=(14, 12))
for k, (name, (fr, mk)) in enumerate(CONDITIONS.items()):
    im = fr[f0].astype(float) / 255
    axes[0, k].imshow(im); axes[0, k].set_title(f"{name}\nframe {f0}: image given to DAM", fontsize=9)
    im = im.copy(); im[mk[f0]] = 0.5 * im[mk[f0]] + 0.5 * np.array([1, 0.2, 0.2])
    axes[1, k].imshow(im); axes[1, k].set_title("red = floor mask given to DAM", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout(); fig.savefig(f"{OUT}/conditions.png", dpi=60); plt.show()

## 3. Describe the floor under each condition (3 runs each)

Same model, same prompt and same settings as stage 4: their `DamCaptioner` and `CaptionConfig`.

In [ ]:
t0 = time.time()
captioner = P.DamCaptioner(cfg.caption, "cuda" if torch.cuda.is_available() else "cpu")
print(f"DAM loaded in {time.time() - t0:.0f} s")

PERSON = r"\b(person|boy|girl|child|kid|individual|man|woman|people|player|he|she|his|her|they|their)\b"
FLOORW = r"\b(floor|trampoline|mat|surface|ground|canvas|bed|bouncing area|platform|carpet)\b"
def verdict(text):
    first = re.split(r"(?<=[.!?])\s", text.strip(), maxsplit=1)[0].lower()     # what the FIRST sentence says it is
    p, q = bool(re.search(PERSON, first)), bool(re.search(FLOORW, first))
    return "PERSON" if p and not q else "FLOOR/SURFACE" if q and not p else "both" if p and q else "other"

results = {}
for name, (fr, mk) in CONDITIONS.items():
    results[name] = []
    for run in range(3):
        d = captioner.describe([fr[f] for f in FR], [mk[f] for f in FR])
        results[name].append({"verdict": verdict(d), "text": d})
        print(f"[{name} · run {run + 1}] {verdict(d):13} | {d[:160]}…")

## 4. Result

In [ ]:
print(f"{'condition':22} | verdicts (first sentence of each run)")
for name, runs in results.items():
    print(f"{name:22} | {', '.join(r['verdict'] for r in runs)}")
lines = ["# Floor experiment: why DAM described the floor (id 0) as a person", "",
         f"Frames {FR}; boy region = ids {BOY_PARTS}; 3 runs per condition; verdict = what the first sentence names.", "",
         "| condition | run 1 | run 2 | run 3 |", "|---|---|---|---|"]
for name, runs in results.items():
    lines.append(f"| {name} | " + " | ".join(r["verdict"] for r in runs) + " |")
lines += ["", "![conditions](conditions.png)", ""]
for name, runs in results.items():
    lines += [f"## {name}", ""] + [f"{i + 1}. {r['text']}" for i, r in enumerate(runs)] + [""]
open(f"{OUT}/README.md", "w").write("\n".join(lines) + "\n")
json.dump({"frames": FR, "results": results}, open(f"{OUT}/results.json", "w"), indent=1)
print("\nsaved", OUT, "\nthen:  git add", OUT, "notebooks/dam_floor_experiment.ipynb && git commit -m 'floor experiment' && git push")

**How to read it:**

| result | meaning |
|---|---|
| (a) says floor | cause 2: the blurry mask let the boy count as part of the object |
| only (b) says floor | cause 3: the boy draws DAM's attention even though the mask is right |
| only (c) says floor | both are needed |
| none say floor | DAM can't describe a large plain surface from a mask: a limit of DAM itself for "stuff" regions |